In [ ]:
import re
from pathlib import Path

def preprocess_text(text: str, mode: str = "novel") -> str:
    """
    Cleans OCR text with different modes:
    - novel:   One sentence per line (default).
    - drama:   Keep dialogues in paragraphs under speaker labels.
    """

    # === Common cleaning ===
    # 1. Remove non-Latin characters (but keep punctuation)
    text = re.sub(r"[^a-zA-Z0-9\s\.\?!\"',;:\-\(\)]", "", text)

    # 2. Remove isolated page numbers
    text = re.sub(r"^\d+\s*$", "", text, flags=re.MULTILINE)

    # 3. Handle OCR hyphenation across lines
    text = re.sub(r"-\s*\n\s*", "", text)

    # 4. Normalize multiple line breaks into single newlines
    text = re.sub(r"\n{2,}", "\n", text)

    # 5. Remove spaces before punctuation
    text = re.sub(r"\s+([\.!?;:,])", r"\1", text)

    if mode == "drama":
        # Merge speaker lines until the next "NAME :"
        # Speaker names are usually ALL CAPS followed by colon
        lines = text.splitlines()
        cleaned = []
        current_speaker = None
        current_dialogue = []

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Match speaker labels like "FIKILE :" or "XHESHILE :"
            match = re.match(r"^([A-Z]+)\s*:", line)
            if match:
                # flush previous speaker block
                if current_speaker and current_dialogue:
                    cleaned.append(f"{current_speaker} : {' '.join(current_dialogue).strip()}")
                # start new block
                current_speaker = match.group(1)
                current_dialogue = [line[len(match.group(0)):].strip()]
            else:
                # continuation of the current speaker's dialogue
                current_dialogue.append(line)

        # flush last speaker
        if current_speaker and current_dialogue:
            cleaned.append(f"{current_speaker} : {' '.join(current_dialogue).strip()}")

        return "\n".join(cleaned)

    elif mode == "novel":
        # Collapse all line breaks into spaces
        text = re.sub(r"\n+", " ", text)

        # Split into sentences at ., ?, !
        sentences = re.split(r'([\.!?][\"\']?)', text)

        cleaned_sentences = []
        current = ""
        for part in sentences:
            current += part.strip() + " "
            if re.fullmatch(r'[\.!?][\"\']?', part.strip()):
                cleaned_sentences.append(current.strip())
                current = ""
        if current.strip():
            cleaned_sentences.append(current.strip())

        return "\n".join(cleaned_sentences)

    else:
        raise ValueError(f"Unsupported mode: {mode}")


In [ ]:
# === Modified to process all files in a folder ===
INPUT_FOLDER = "input_folder"  # Change this to your folder path
OUTPUT_FILE = "combined_corpus.txt"

input_dir = Path.cwd() / INPUT_FOLDER
out_dir = Path.cwd() / "final_corpus"
out_dir.mkdir(parents=True, exist_ok=True)

all_processed_text = []

In [ ]:
# Process all .txt files in the input folder
for text_file in input_dir.glob("*.txt"):
    print(f"Processing: {text_file.name}")
    
    with open(text_file, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Change mode to "drama" or "novel" as needed
    processed_text = preprocess_text(raw_text, mode="novel")
    all_processed_text.append(processed_text)

In [ ]:
# Combine all processed text with separators
combined_text = "\n\n" + "="*50 + "\n\n".join(all_processed_text)

# Write to single output file
out_path = out_dir / OUTPUT_FILE
with open(out_path, "w", encoding="utf-8") as f:
    f.write(combined_text)

print(f"Processing complete! Combined file saved as: {out_path}")
print(f"Processed {len(all_processed_text)} files")